1. CSV → Pandas → Data Cleaning → SQL Database → Report

In [ ]:

# Data Engineering Practicals
   #   Practical-9
# Name: Insiya Shoeb Bobde
# Rollno: 06
# Student-id: 5115304


import pandas as pd

data = {
    "Customer": ["John", "Mary", "John", "David", None],
    "Product": ["Laptop", "Phone", "Laptop", "Tablet", "Phone"],
    "Sales": [1000, 500, 1000, 700, None],
    "Quantity": [1, 2, 1, 3, 2]
}

df = pd.DataFrame(data)

df.to_csv("sales_data.csv", index=False)

print("CSV created successfully!")
df

CSV created successfully!


,Customer,Product,Sales,Quantity
0,John,Laptop,1000.0,1
1,Mary,Phone,500.0,2
2,John,Laptop,1000.0,1
3,David,Tablet,700.0,3
4,None,Phone,NaN,2


In [2]:
df = pd.read_csv("sales_data.csv")

print("Original Data:")
print(df)

Original Data:
  Customer Product   Sales  Quantity
0     John  Laptop  1000.0         1
1     Mary   Phone   500.0         2
2     John  Laptop  1000.0         1
3    David  Tablet   700.0         3
4      NaN   Phone     NaN         2


In [3]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
Customer    1
Product     0
Sales       1
Quantity    0
dtype: int64


In [4]:
df["Customer"] = df["Customer"].fillna("Unknown")
df["Sales"] = df["Sales"].fillna(df["Sales"].mean())

print("Cleaned Data:")
print(df)

Cleaned Data:
  Customer Product   Sales  Quantity
0     John  Laptop  1000.0         1
1     Mary   Phone   500.0         2
2     John  Laptop  1000.0         1
3    David  Tablet   700.0         3
4  Unknown   Phone   800.0         2


In [5]:
df = df.drop_duplicates()

print("After removing duplicates:")
print(df)

After removing duplicates:
  Customer Product   Sales  Quantity
0     John  Laptop  1000.0         1
1     Mary   Phone   500.0         2
3    David  Tablet   700.0         3
4  Unknown   Phone   800.0         2


In [6]:
import sqlite3

connection = sqlite3.connect("sales_database.db")

df.to_sql(
    "sales",
    connection,
    if_exists="replace",
    index=False
)

print("Data stored in SQL database successfully!")

Data stored in SQL database successfully!


In [7]:
query = """
SELECT Product, SUM(Sales) AS Total_Sales
FROM sales
GROUP BY Product
"""

report = pd.read_sql_query(query, connection)

print("Sales Report:")
print(report)

Sales Report:
  Product  Total_Sales
0  Laptop       1000.0
1   Phone       1300.0
2  Tablet        700.0


In [8]:
connection.close()

print("Pipeline completed successfully!")

Pipeline completed successfully!


2. ETL pipeline for an e-commerce dataset

In [9]:
import pandas as pd

customers = pd.DataFrame({
    "customer_id": [1, 2, 3],
    "name": ["John", "Mary", "David"],
    "email": ["john@gmail.com", "mary@gmail.com", "david@gmail.com"]
})

products = pd.DataFrame({
    "product_id": [101, 102, 103],
    "product": ["Laptop", "Phone", "Tablet"],
    "price": [1000, 500, 700]
})

orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003],
    "customer_id": [1, 2, 3],
    "product_id": [101, 102, 103],
    "quantity": [1, 2, 1]
})

payments = pd.DataFrame({
    "payment_id": [501, 502, 503],
    "order_id": [1001, 1002, 1003],
    "payment_status": ["Paid", "Paid", "Pending"]
})

print("Data extracted successfully!")

Data extracted successfully!


In [10]:
order_data = orders.merge(
    products,
    on="product_id"
)

order_data["total_amount"] = (
    order_data["price"] * order_data["quantity"]
)

print("Transformed Order Data:")
print(order_data)

Transformed Order Data:
   order_id  customer_id  product_id  quantity product  price  total_amount
0      1001            1         101         1  Laptop   1000          1000
1      1002            2         102         2   Phone    500          1000
2      1003            3         103         1  Tablet    700           700


In [11]:
order_data = order_data.merge(
    customers,
    on="customer_id"
)

print("Customer + Order Data:")
print(order_data)

Customer + Order Data:
   order_id  customer_id  product_id  quantity product  price  total_amount  \
0      1001            1         101         1  Laptop   1000          1000   
1      1002            2         102         2   Phone    500          1000   
2      1003            3         103         1  Tablet    700           700   

    name            email  
0   John   john@gmail.com  
1   Mary   mary@gmail.com  
2  David  david@gmail.com  


In [12]:
order_data = order_data.merge(
    payments,
    on="order_id"
)

print("Complete E-commerce Data:")
print(order_data)

Complete E-commerce Data:
   order_id  customer_id  product_id  quantity product  price  total_amount  \
0      1001            1         101         1  Laptop   1000          1000   
1      1002            2         102         2   Phone    500          1000   
2      1003            3         103         1  Tablet    700           700   

    name            email  payment_id payment_status  
0   John   john@gmail.com         501           Paid  
1   Mary   mary@gmail.com         502           Paid  
2  David  david@gmail.com         503        Pending  


In [13]:
import sqlite3

connection = sqlite3.connect("ecommerce.db")

customers.to_sql(
    "customers",
    connection,
    if_exists="replace",
    index=False
)

products.to_sql(
    "products",
    connection,
    if_exists="replace",
    index=False
)

orders.to_sql(
    "orders",
    connection,
    if_exists="replace",
    index=False
)

payments.to_sql(
    "payments",
    connection,
    if_exists="replace",
    index=False
)

order_data.to_sql(
    "order_report",
    connection,
    if_exists="replace",
    index=False
)

print("ETL pipeline completed!")

ETL pipeline completed!


In [14]:
query = """
SELECT product,
       SUM(quantity) AS total_quantity,
       SUM(total_amount) AS total_sales
FROM order_report
GROUP BY product
"""

report = pd.read_sql_query(query, connection)

print("E-commerce Sales Report:")
print(report)

E-commerce Sales Report:
  product  total_quantity  total_sales
0  Laptop               1         1000
1   Phone               2         1000
2  Tablet               1          700


In [15]:
connection.close()

3. Historical + newly arriving data pipeline

In [16]:
import pandas as pd

historical_data = pd.DataFrame({
    "order_id": [1, 2, 3],
    "customer": ["John", "Mary", "David"],
    "sales": [1000, 500, 700]
})

historical_data

,order_id,customer,sales
0,1,John,1000
1,2,Mary,500
2,3,David,700


In [17]:
new_data = pd.DataFrame({
    "order_id": [4, 5],
    "customer": ["Alice", "Bob"],
    "sales": [800, 600]
})

new_data

,order_id,customer,sales
0,4,Alice,800
1,5,Bob,600


In [18]:
combined_data = pd.concat(
    [historical_data, new_data],
    ignore_index=True
)

print("Combined Data:")
print(combined_data)

Combined Data:
   order_id customer  sales
0         1     John   1000
1         2     Mary    500
2         3    David    700
3         4    Alice    800
4         5      Bob    600


In [19]:
combined_data = combined_data.drop_duplicates(
    subset=["order_id"]
)

print("Data after removing duplicates:")
print(combined_data)

Data after removing duplicates:
   order_id customer  sales
0         1     John   1000
1         2     Mary    500
2         3    David    700
3         4    Alice    800
4         5      Bob    600


In [20]:
import sqlite3

connection = sqlite3.connect("historical_new_data.db")

combined_data.to_sql(
    "orders",
    connection,
    if_exists="replace",
    index=False
)

print("Historical and new data loaded into database!")

Historical and new data loaded into database!


In [21]:
query = """
SELECT customer,
       SUM(sales) AS total_sales
FROM orders
GROUP BY customer
"""

final_report = pd.read_sql_query(
    query,
    connection
)

print("Final Report:")
print(final_report)

Final Report:
  customer  total_sales
0    Alice          800
1      Bob          600
2    David          700
3     John         1000
4     Mary          500


In [22]:
connection.close()

print("Historical + New Data Pipeline Completed!")

Historical + New Data Pipeline Completed!
